# Notebook para garantir logo branco e fundo branco sempre
Este notebook ajuda a inspecionar, converter e testar o ícone PWA branco com fundo branco, incluindo geração de PNG/SVG e validação de metadados.

## 1. Importar bibliotecas
Instalar/importar Pillow, numpy, cairosvg (opcional), lxml, IPython.display e pytest para testes.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from IPython.display import display, HTML

print("Bibliotecas importadas com sucesso")

## 2. Carregar e inspecionar logos
Carregar arquivos raster (PNG/JPG) e SVG; mostrar tamanho, modos de cor, e visualizar dentro do notebook.

In [ ]:
root = Path('.')
logo_white_path = root / 'img' / 'logo-white.png'
logo_black_path = root / 'img' / 'logo-black.png'

for path in [logo_white_path, logo_black_path]:
    if path.exists():
        with Image.open(path) as img:
            print(f"{path.name}: size={img.size}, mode={img.mode}, format={img.format}")
            display(img)
    else:
        print(f"Arquivo não encontrado: {path}")

## 3. Converter logo raster para branco
Usar Pillow/numpy para localizar pixels não transparentes e definir cor para branca mantendo transparência; normalizar alfa.

In [ ]:
def convert_to_white_icon(input_path: Path, output_path: Path):
    with Image.open(input_path) as img:
        img = img.convert('RGBA')
        data = np.array(img)
        alpha = data[..., 3] / 255.0
        non_transparent = alpha > 0
        data[..., :3][non_transparent] = [255, 255, 255]
        white_img = Image.fromarray(data, 'RGBA')
        white_img.save(output_path, optimize=True)
    return output_path

output_path = root / 'img' / 'logo-white-generated.png'
convert_to_white_icon(logo_black_path, output_path)
print(f"Imagem salva em: {output_path}")
with Image.open(output_path) as img:
    display(img)

## 4. Converter logo SVG para branco
Parsar SVG com lxml ou editar strings para sobrescrever atributos `fill`/`stroke` para `#ffffff`; gerar SVG resultante e validar.

In [ ]:
from lxml import etree

svg_path = root / 'img' / 'logo-white.svg'
svg_output = root / 'img' / 'logo-white-generated.svg'

if svg_path.exists():
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(str(svg_path), parser)
    root_elem = tree.getroot()
    for elem in root_elem.xpath('//*'):
        if 'fill' in elem.attrib:
            elem.attrib['fill'] = '#ffffff'
        if 'stroke' in elem.attrib:
            elem.attrib['stroke'] = '#ffffff'
    tree.write(str(svg_output), pretty_print=True, xml_declaration=True, encoding='utf-8')
    print(f"SVG convertido salvo em: {svg_output}")
else:
    print(f"SVG não encontrado: {svg_path}")

## 5. Exportar versões otimizadas (PNG, SVG, WebP)
Salvar PNG com transparência, PNG com fundo branco preenchido, exportar WebP lossless, e otimizar SVG.

In [ ]:
png_output = root / 'img' / 'logo-white-whitebg.png'
webp_output = root / 'img' / 'logo-white.webp'

if logo_white_path.exists():
    with Image.open(logo_white_path) as img:
        img = img.convert('RGBA')
        white_bg = Image.new('RGB', img.size, (255, 255, 255))
        white_bg.paste(img, mask=img.split()[3])
        white_bg.save(png_output, optimize=True)
        img.save(webp_output, format='WEBP', lossless=True)
        print(f"PNG com fundo branco salvo em: {png_output}")
        print(f"WebP lossless salvo em: {webp_output}")
else:
    print('logo-white.png não encontrado para exportação')

if svg_output.exists():
    with open(svg_output, 'r', encoding='utf-8') as f:
        svg_text = f.read()
        print('SVG convertido tem tamanho', len(svg_text))
else:
    print('SVG convertido não encontrado')

## 6. Forçar fundo branco via CSS/HTML
Incluir snippets HTML/CSS que definem `background-color: #ffffff !important;` e regras para imagens `img/svg`.

In [ ]:
html_snippet = '''
<style>
  html, body {
    background-color: #ffffff !important;
    color: #000;
  }
  img.pwa-logo, svg.pwa-logo {
    background-color: #ffffff !important;
    border-radius: 20%;
  }
  .pwa-icon {
    background-color: #ffffff !important;
  }
</style>
'''
print(html_snippet)
HTML(html_snippet)

## 7. Visualizar e testar no notebook
Renderizar exemplos HTML no notebook e mostrar imagens geradas; testar variações de contexto (dark mode, fundo colorido).

In [ ]:
html_example = '''
<div style="background: #ffffff; padding: 16px; border: 1px solid #ddd; display:inline-block;">
  <img src="img/logo-white.png" alt="Logo branco" width="128" height="128" class="pwa-logo">
  <p>Logo branco sobre branco</p>
</div>
'''
display(HTML(html_example))

## 8. Testes automatizados (validação de pixels/CSS)
Escrever testes pytest para verificar que pixels esperados são `#ffffff`, que SVG contém `fill="#ffffff"`, e que HTML contém regras CSS exigidas.

In [ ]:
def validate_white_png(path: Path):
    with Image.open(path) as img:
        img = img.convert('RGBA')
        data = np.array(img)
        alpha = data[..., 3] > 0
        rgb = data[..., :3]
        non_transparent = rgb[alpha]
        if non_transparent.size == 0:
            return True
        return np.all(non_transparent == 255)

print('logo-white.png válidos?', validate_white_png(logo_white_path))
print('logo-white-generated.png válidos?', validate_white_png(output_path))

## 9. Integração com VSCode
Comandos para rodar pytest no terminal integrado, tasks.json sugerido e instruções para abrir arquivos gerados e ver saída no painel Output/Terminal.

Use os seguintes comandos no terminal integrado:

- `python -m pip install pillow numpy lxml pytest cairosvg`
- `python -m pytest pwa-white-icon.ipynb` (usando pytest-notebook ou conversão de testes)

Sugestão de `tasks.json`:
```json
{
  "version": "2.0.0",
  "tasks": [
    {
      "label": "Run PWA white icon tests",
      "type": "shell",
      "command": "python -m pytest",
      "args": ["-q"],
      "group": {
        "kind": "test",
        "isDefault": true
      },
      "presentation": {
        "echo": true,
        "reveal": "always"
      }
    }
  ]
}
```

Abra o notebook no editor e use a célula de código para visualizar as imagens e testar as conversões.